In [1]:
import pandas as pd
import numpy as np
import psycopg2
import sqlalchemy as db
from sqlalchemy import create_engine
import yaml

In [2]:
with open('.\\config.yml', 'r') as f:
    config = yaml.safe_load(f)
    config_co = config['mensajeria']
    config_etl = config['ETL_PRO']

url_co = (f"{config_co['drivername']}://{config_co['user']}:{config_co['password']}@{config_co['host']}:"
          f"{config_co['port']}/{config_co['dbname']}")
url_etl = (f"{config_etl['drivername']}://{config_etl['user']}:{config_etl['password']}@{config_etl['host']}:"
           f"{config_etl['port']}/{config_etl['dbname']}")

mensajeria = create_engine(url_co)
etl_conn   = create_engine(url_etl)

In [3]:
novedades_raw = pd.read_sql_table('mensajeria_novedadesservicio', mensajeria)

dim_novedades  = pd.read_sql_table('dim_novedades',  etl_conn)
dim_mensajero  = pd.read_sql_table('dim_mensajero',  etl_conn)
dim_fecha      = pd.read_sql_table('dim_fecha',      etl_conn)
trans_servicio = pd.read_sql_table('trans_servicio', etl_conn)

In [4]:
novedades_raw.head()

,id,fecha_novedad,tipo_novedad_id,descripcion,servicio_id,es_prueba,mensajero_id
0,4,2023-11-30 05:00:00+00:00,1,A,51,True,7
1,5,2023-11-30 05:00:00+00:00,1,Halo,51,True,7
2,6,2023-11-30 05:00:00+00:00,1,A,51,True,7
3,7,2023-11-30 05:00:00+00:00,1,B,51,True,7
4,8,2023-11-30 05:00:00+00:00,1,A,51,True,7


In [5]:
novedades_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 5208 entries, 0 to 5207
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype              
---  ------           --------------  -----              
 0   id               5208 non-null   int64              
 1   fecha_novedad    5208 non-null   datetime64[us, UTC]
 2   tipo_novedad_id  5208 non-null   int64              
 3   descripcion      5208 non-null   str                
 4   servicio_id      5208 non-null   int64              
 5   es_prueba        5208 non-null   bool               
 6   mensajero_id     5208 non-null   int64              
dtypes: bool(1), datetime64[us, UTC](1), int64(4), str(1)
memory usage: 249.3 KB


In [6]:
novedades_raw = novedades_raw[novedades_raw['es_prueba'] == False].copy()

novedades_raw = novedades_raw.rename(columns={'id': 'novedad_id'})

dim_fecha['fecha_completa'] = pd.to_datetime(dim_fecha['fecha_completa']).dt.normalize()
novedades_raw['fecha_novedad'] = pd.to_datetime(novedades_raw['fecha_novedad']).dt.tz_convert(None).dt.normalize()

val_a = novedades_raw['fecha_novedad'].iloc[0]
val_b = dim_fecha['fecha_completa'].iloc[0]
print(repr(val_a), repr(val_b))
print(type(val_a), type(val_b))

novedades_raw.head()

Timestamp('2023-12-28 00:00:00') Timestamp('2020-01-01 00:00:00')
<class 'pandas.Timestamp'> <class 'pandas.Timestamp'>


,novedad_id,fecha_novedad,tipo_novedad_id,descripcion,servicio_id,es_prueba,mensajero_id
13,19,2023-12-28,2,X,68,False,7
14,20,2023-12-28,1,X,68,False,7
15,21,2023-12-28,2,,68,False,7
16,22,2023-12-28,2,no puedo continuar,73,False,7
19,25,2024-01-04,1,DEMORA,85,False,7


In [7]:
hecho_novedades = novedades_raw.merge(
    dim_novedades[['novedad_id', 'key_novedad']],
    on='novedad_id',
    how='left'
)

hecho_novedades = hecho_novedades.merge(
    dim_mensajero[['id_operaciones', 'key_mensajero']],
    left_on='mensajero_id',
    right_on='id_operaciones',
    how='left'
).rename(columns={'key_mensajero': 'key_dim_mensajero'})

hecho_novedades = hecho_novedades.merge(
    trans_servicio[['servicio_id', 'key_trans_servicio']],
    on='servicio_id',
    how='left'
)

hecho_novedades = hecho_novedades.merge(
    dim_fecha[['fecha_completa', 'key_fecha']],
    left_on='fecha_novedad',
    right_on='fecha_completa',
    how='left'
).rename(columns={'key_fecha': 'key_dim_fecha_novedad'})
hecho_novedades.drop(columns=['fecha_completa'], inplace=True)

hecho_novedades['key_novedad'].value_counts()

key_novedad
14    1
15    1
16    1
17    1
20    1
26    1
46    1
47    1
Name: count, dtype: int64

In [8]:
hecho_novedades['cantidad_novedades'] = 1

cols_to_drop = [
    'novedad_id',
    'mensajero_id',
    'tipo_novedad_id',
    'descripcion',
    'servicio_id',
    'es_prueba',
    'fecha_novedad',
    'id_operaciones'
]
hecho_novedades.drop(
    columns=[c for c in cols_to_drop if c in hecho_novedades.columns],
    inplace=True
)


key_cols = ['key_novedad', 'key_dim_mensajero', 'key_trans_servicio', 'key_dim_fecha_novedad']
for col in key_cols:
    hecho_novedades[col] = hecho_novedades[col].astype('Int64')

hecho_novedades.info()

<class 'pandas.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 5 columns):
 #   Column                 Non-Null Count  Dtype
---  ------                 --------------  -----
 0   key_novedad            8 non-null      Int64
 1   key_dim_mensajero      8 non-null      Int64
 2   key_trans_servicio     7 non-null      Int64
 3   key_dim_fecha_novedad  8 non-null      Int64
 4   cantidad_novedades     8 non-null      int64
dtypes: Int64(4), int64(1)
memory usage: 484.0 bytes


In [9]:
hecho_novedades.head()

,key_novedad,key_dim_mensajero,key_trans_servicio,key_dim_fecha_novedad,cantidad_novedades
0,14,14,51,20231228,1
1,15,14,51,20231228,1
2,16,14,51,20231228,1
3,17,14,54,20231228,1
4,20,14,55,20240104,1


In [10]:
hecho_novedades.to_sql('hecho_novedades', etl_conn, if_exists='replace', index=False)
print(f'hecho_novedades cargada correctamente: {hecho_novedades.shape[0]} filas, {hecho_novedades.shape[1]} columnas')

hecho_novedades cargada correctamente: 8 filas, 5 columnas
